In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings(action='ignore')

In [2]:
X = pd.read_csv('./data/X.csv', encoding='euc-kr')
y = pd.read_csv('./data/y.csv', encoding='euc-kr')

In [3]:
X.head()

,cust_id,총구매액,최대구매액,환불금액,주구매상품,주구매지점,내점일수,내점당구매건수,주말방문비율,구매주기
0,0,68282840,11264000,6860000.0,기타,강남점,19,3.894737,0.527027,17
1,1,2136000,2136000,300000.0,스포츠,잠실점,2,1.500000,0.000000,1
2,2,3197000,1639000,NaN,남성 캐주얼,관악점,2,2.000000,0.000000,1
3,3,16077620,4935000,NaN,기타,광주점,18,2.444444,0.318182,16
4,4,29050000,24000000,NaN,보석,본 점,2,1.500000,0.000000,85


In [4]:
y.head()

,cust_id,gender
0,0,0
1,1,0
2,2,1
3,3,1
4,4,0


In [5]:
# 1) cust_id 중복 개수
dup_x = X["cust_id"].duplicated().sum()
dup_y = y["cust_id"].duplicated().sum()

print("X(cust_id) 중복 개수:", dup_x)
print("y(cust_id) 중복 개수:", dup_y)

X(cust_id) 중복 개수: 0
y(cust_id) 중복 개수: 0


In [6]:
y['gender'].value_counts() #0 여성, 1 남성

gender
0    2184
1    1316
Name: count, dtype: int64

In [7]:
y["gender"].value_counts(normalize=True, dropna=False).sort_index() * 100

gender
0    62.4
1    37.6
Name: proportion, dtype: float64

# 데이터 탐색

In [8]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   cust_id  3500 non-null   int64  
 1   총구매액     3500 non-null   int64  
 2   최대구매액    3500 non-null   int64  
 3   환불금액     1205 non-null   float64
 4   주구매상품    3500 non-null   str    
 5   주구매지점    3500 non-null   str    
 6   내점일수     3500 non-null   int64  
 7   내점당구매건수  3500 non-null   float64
 8   주말방문비율   3500 non-null   float64
 9   구매주기     3500 non-null   int64  
dtypes: float64(3), int64(5), str(2)
memory usage: 273.6 KB


In [9]:
X.describe()

,cust_id,총구매액,최대구매액,환불금액,내점일수,내점당구매건수,주말방문비율,구매주기
count,3500.000000,3.500000e+03,3.500000e+03,1.205000e+03,3500.000000,3500.000000,3500.000000,3500.000000
mean,1749.500000,9.191925e+07,1.966424e+07,2.407822e+07,19.253714,2.834963,0.307246,20.958286
std,1010.507298,1.635065e+08,3.199235e+07,4.746453e+07,27.174942,1.912368,0.289752,24.748682
min,0.000000,-5.242152e+07,-2.992000e+06,5.600000e+03,1.000000,1.000000,0.000000,0.000000
25%,874.750000,4.747050e+06,2.875000e+06,2.259000e+06,2.000000,1.666667,0.027291,4.000000
50%,1749.500000,2.822270e+07,9.837000e+06,7.392000e+06,8.000000,2.333333,0.256410,13.000000
75%,2624.250000,1.065079e+08,2.296250e+07,2.412000e+07,25.000000,3.375000,0.448980,28.000000
max,3499.000000,2.323180e+09,7.066290e+08,5.637530e+08,285.000000,22.083333,1.000000,166.000000


# 데이터 전처리

In [10]:
# X에서 cust_id 제거
X = X.drop(columns=["cust_id"])

# y에서 cust_id 제거(타깃만 남김)
y = y["gender"]

In [11]:
missing_count = X.isna().sum().sort_values(ascending=False)
missing_count

환불금액       2295
총구매액          0
최대구매액         0
주구매상품         0
주구매지점         0
내점일수          0
내점당구매건수       0
주말방문비율        0
구매주기          0
dtype: int64

In [12]:
X = X.fillna(0)

In [13]:
cat_cols = X.select_dtypes(include=['object'])
col_list = cat_cols.columns.to_list() #chaining
col_list

['주구매상품', '주구매지점']

In [14]:
for col in col_list:
  # print(col)
  print(X[col].nunique())
  print(X[col].value_counts().head()) #범주형변수의 값의 종류 

42
주구매상품
기타      595
가공식품    546
농산물     339
화장품     264
시티웨어    213
Name: count, dtype: int64
24
주구매지점
본  점    1077
잠실점      474
분당점      436
부산본점     245
영등포점     241
Name: count, dtype: int64


# 원핫인코딩, 라벨인코딩 결과 비교

In [15]:
# 필요한 라이브러리 불러오기
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [16]:
# 범주형 변수 인코딩 (LabelEncoder 사용)
le = LabelEncoder()

for col in cat_cols:
    X[col] = le.fit_transform(X[col])
    print(f"{col} 인코딩 완료")

X.head()

주구매상품 인코딩 완료
주구매지점 인코딩 완료


,총구매액,최대구매액,환불금액,주구매상품,주구매지점,내점일수,내점당구매건수,주말방문비율,구매주기
0,68282840,11264000,6860000.0,5,0,19,3.894737,0.527027,17
1,2136000,2136000,300000.0,21,19,2,1.500000,0.000000,1
2,3197000,1639000,0.0,6,1,2,2.000000,0.000000,1
3,16077620,4935000,0.0,5,2,18,2.444444,0.318182,16
4,29050000,24000000,0.0,15,8,2,1.500000,0.000000,85


# 스케일링

In [17]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [18]:
scaler = StandardScaler() #클래스 > import
X_scaled = scaler.fit_transform(X)
# X_scaled[:3]
X_scaled #numpy의 array

array([[-0.14458009, -0.26260786, -0.04750476, ...,  0.55424743,
         0.75862274, -0.15996211],
       [-0.54918957, -0.54796686, -0.26546133, ..., -0.69816782,
        -1.06053002, -0.80655356],
       [-0.5426996 , -0.56350404, -0.27542885, ..., -0.43667453,
        -1.06053002, -0.80655356],
       ...,
       [-0.56179637, -0.61239772, -0.27542885, ..., -0.95966112,
        -1.06053002, -0.84696552],
       [-0.55078606, -0.58348042, -0.27542885, ..., -0.95966112,
        -1.06053002,  0.72910114],
       [ 1.04709431,  0.46792117, -0.07697541, ..., -0.21646965,
         0.55277658, -0.5236698 ]], shape=(3500, 9))

# 학습/테스트 데이터 분할

In [19]:
# 학습/테스트 데이터 분리 (8:2 비율)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y #y가 불균형이므로 설정
)
X_train.shape, X_test.shape

((2800, 9), (700, 9))

# 모델링 - LogisticRegression

In [20]:
from sklearn.linear_model import LogisticRegression

In [21]:
X_train, X_test, y_train, y_test

(array([[ 0.08226136,  0.16680616, -0.27542885, ..., -0.54127184,
          1.70085057, -0.32160997],
        [-0.553924  , -0.58191732, -0.27542885, ..., -0.95966112,
          0.66533285,  1.77981225],
        [-0.49402456, -0.36073909, -0.27542885, ..., -0.43667453,
          0.23386713,  1.49692849],
        ...,
        [-0.5511921 , -0.58660662, -0.27542885, ...,  1.39377854,
          1.13602273,  2.79011139],
        [ 0.67521934, -0.2658591 , -0.26778708, ...,  1.8232033 ,
          0.05664682, -0.72572963],
        [ 4.5535857 ,  1.60485543,  0.44223302, ...,  0.04344431,
          0.21932335, -0.6449057 ]], shape=(2800, 9)),
 array([[-0.55522077, -0.57879113, -0.27542885, ..., -0.95966112,
          2.39119572, -0.84696552],
        [-0.52677136, -0.46624814, -0.21402889, ..., -0.78533225,
          1.52826429, -0.60449373],
        [-0.53922524, -0.54665386, -0.27542885, ..., -0.69816782,
         -1.06053002, -0.36202194],
        ...,
        [ 0.09294307,  0.26609408,  0

In [22]:
lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced")

In [23]:
lr_model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

# 예측 성능 확인해보기 - LogisticRegression

In [24]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [25]:
# 예측
lr_pred = lr_model.predict(X_test)

# 평가함수 evaluate_classifier()

In [26]:
def evaluate_classifier(y_true, y_pred):
    # 정확도: 전체 정답 비율
    acc = accuracy_score(y_true, y_pred)

    # 정밀도/재현율/F1: 기본적으로 positive label = 1 기준
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    # 혼동행렬: [[TN, FP], [FN, TP]]
    cm = confusion_matrix(y_true, y_pred)

    print("Accuracy :", round(acc, 4))
    print("Precision:", round(prec, 4))
    print("Recall   :", round(rec, 4))
    print("F1       :", round(f1, 4))
    print("\nConfusion Matrix")
    print(cm)

In [27]:
evaluate_classifier(y_test, lr_pred)

Accuracy : 0.55
Precision: 0.4333
Recall   : 0.6426
F1       : 0.5176

Confusion Matrix
[[216 221]
 [ 94 169]]
